# Modele de prediction

- **Objectif:** 

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import average_precision_score, precision_score, recall_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, log_loss
from config import DATASET

## Preparation des donnees

### Chargement des donnees

In [ ]:
data = pd.read_csv(DATASET).sort_values('date_debut').reset_index(drop=True)
data.head()

In [ ]:
# Separation des variables
y = data.pop('churn')
x = data.drop(['id_inscription','id_etudiant','date_debut'],axis='columns')

### Train/test split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.25,shuffle=False)
print("Taille totale")
print(len(x))
print("taille de training set")
print(len(x_train))
print("taille de test set")
print(len(x_test))

### Transofrmation des donnees

In [ ]:
nominal = ['region','niveau_scolaire','annee_scolaire']
num = ['age','nb_inscriptions_precedentes', 'total_volume','days_since_last_activity', 'n_evaluations']

preprocess = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'),nominal),
        ('scale', StandardScaler(), num)
    ],
    remainder='passthrough'
)

x_train = preprocess.fit_transform(x_train)
x_test = preprocess.transform(x_test)

## Modeles de prediction

### Regression logistique

1. Construction

In [ ]:
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=10)
log_reg.fit(x_train,y_train)

2. Evaluation

In [ ]:
y_pred = log_reg.predict(x_test)
print(log_reg.score(x_test,y_test))

y_prob = log_reg.predict_proba(x_test)[:,1]

# Matrice de confusion
cm =confusion_matrix(y_test,y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0,1])
disp.plot(cmap='Reds')
plt.title('Matrice de confusion')
plt.show()

In [ ]:
print(classification_report(y_test,y_pred,digits=4))
print(f"PR-AUC: {average_precision_score(y_test, y_prob):.4f}")
print(f"Log-Loss: {log_loss(y_test, y_prob):.4f}")

### Arbre de decision

In [ ]:
tree = DecisionTreeClassifier(random_state=10, class_weight='balanced')
tree.fit(x_train,y_train)

### SVM

In [ ]:
svm = SVC(random_state=10, class_weight='balanced')
svm.fit(x_train,y_train)